#Set-ExecutionPolicy -ExecutionPolicy Bypass -Scope CurrentUser

# 1. Preparation

## install require packages
geopandas, matplotlib, osmnx, contextily, folium, mapclassify 

py -3.11 -V
py install 3.11
py -3.11 -m genv311

In [ ]:
# import require packages
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import contextily as ctx

In [ ]:
gdf_hoods = gpd.read_file(r"data\mahallat.shp")
gdf_popblocks = gpd.read_file('data/pop/tehran_popblocks_95.shp')

In [ ]:
place = 'Tehran, Iran'
tags = {'amenity': 'hospital'}
hospitals = ox.features_from_place(place, tags)


In [ ]:
hospitals.geom_type.value_counts()


In [ ]:
gdf_hospitals = hospitals[hospitals.geom_type == 'Polygon']

In [ ]:
gdf_hospitals.sample(1).T

## 2. 2SFCA Model

In [ ]:
supply = gdf_hospitals
demand = gdf_popblocks
agg_zones = gdf_hoods
radius = 1500 # meters

In [ ]:
supply=supply.to_crs(demand.crs)

In [ ]:
supply.crs

### 2.1 Step 1 (Supply-to-Demand Ratio at Services)

In [ ]:
Rj_scores = []
for idx, row in supply.iterrows():
  catch = demand[demand.geometry.centroid.distance(row.geometry.centroid)> radius]
  demand_sum = catch['POPULATION'].sum()
  if demand_sum > 0:
    ratio = row.geometry.area/demand_sum
  else:
    ratio = 0
  Rj_scores.append(ratio)

supply['Rj'] = Rj_scores
supply['Rj']


### 2.2 Step 2 (Accessibility Score at Demand Points)

In [ ]:
arr_2sfca = []

for geom in demand.geometry:
    catchment = supply[supply.geometry.centroid.distance(geom.centroid) < radius]
    arr_2sfca.append(catchment['Rj'].sum())

demand['acc_2sfca'] = arr_2sfca


In [ ]:
demand['acc_2sfca'] 

In [ ]:
## 3. Aggregate and Analyze

In [ ]:
demand_hoods = gpd.sjoin(demand, agg_zones, how="inner", predicate="intersects")

demand_hoods.sample(1).T

In [ ]:
demand_agg = demand_hoods.groupby('CODE').agg({
    'POPULATION': 'sum',
    'acc_2sfca': 'mean'
})

In [ ]:
agg_zones_2sfca = agg_zones.merge(demand_agg, on = 'CODE', how='left')

In [ ]:
ax = agg_zones_2sfca.plot(
    column='acc_2sfca',
    cmap='viridis',
    figsize=(10, 15),
    legend = True,
    legend_kwds={
        'orientation': 'vertical',
        'shrink': 0.2,  # smaller legend
        'pad': 0.01,    # tighter spacing
        
    },
    missing_kwds={'color': 'gray'}
    )
ax.set_title("2SFCA Accessibility")    
ax.set_axis_off()

plt.show()



In [ ]:
# Cusomize plot 

fig, ax = plt.subplots(figsize=(10, 15))

# Plot base map
agg_zones_2sfca.plot(
    column='acc_2sfca',
    cmap='viridis',
    linewidth=0.3,
    edgecolor='white',
    ax=ax,
    legend=True,
    legend_kwds={
        'orientation': 'horizontal',
        'shrink': 0.4,  # smaller legend
        'pad': 0.01,    # tighter spacing
    },
    missing_kwds={'color': '#d9d9d9'}
)
# --- Add OSM basemap underneath ---
# Ensure CRS is Web Mercator (EPSG:3857) before adding tiles
if agg_zones_2sfca.crs != "EPSG:3857":
    agg_zones_2sfca = agg_zones_2sfca.to_crs(epsg=3857)

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)

ax.set_title(
    "2SFCA Accessibility",
    loc='left',
    fontsize=24,
    fontweight='bold',   # <— makes it bold
    pad=8
)
ax.set_axis_off()

# --- Move the colorbar INSIDE the map area ---
fig.canvas.draw()
cax = fig.axes[1]

# These coordinates keep it *inside* the map (bottom-left)
# Adjust slightly if needed depending on your layout
bbox = ax.get_position()
cax.set_position([
    bbox.x0 + 0.05,      # shift a bit right from left edge
    bbox.y0 + 0.02,      # slightly above the bottom edge
    0.3,                 # width
    0.015                # height
])

plt.show()


In [ ]:
fig.canvas.draw_idle()
plt.pause(1)  # short delay lets tiles load
fig.savefig('2sfca.png', bbox_inches='tight')


In [ ]:
m = agg_zones_2sfca.explore(column='acc_2sfca')

m.save('2scfa.html')